# Акустическая диагностика промышленного оборудования

## MIMII Dataset — Anomaly Detection & Defect Classification

**Задачи:**
1. Обнаружение аномалий (Convolutional Autoencoder) — обучение только на нормальных звуках
2. Классификация типа дефекта (MLP на латентных признаках автоэнкодера)
3. Оценка устойчивости к шуму (SNR: -6, 0, 6 dB)
4. Бонус: One-Class SVM на MFCC

**Датасет:** [MIMII (Hitachi, 2019)](https://zenodo.org/record/3384388)

In [1]:
import sys
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from src.config import *
from src.download import download_mimii
from src.dataset import (
    discover_files, build_datasets, load_audio, window_signal,
    extract_mel_spectrogram, extract_mfcc, MelSpectrogramDataset,
)
from src.autoencoder import ConvAutoencoder, train_autoencoder, compute_threshold
from src.classifier import DefectClassifier, extract_latent_features, train_classifier
from src.evaluate import evaluate_anomaly_detection, evaluate_classifier, evaluate_snr_robustness, partial_auc
from src.visualize import (
    plot_reconstruction_errors, plot_roc_curve, plot_tsne,
    plot_confusion_matrix, plot_snr_comparison,
    plot_spectrogram_comparison, plot_training_history,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

Device: cpu
PyTorch: 2.12.1+cpu


## 1. Загрузка данных

Скачиваем MIMII dataset (fan). При необходимости можно загрузить все типы машин,
раскомментировав соответствующую строку.

In [ ]:
# Скачать данные для одного типа машины (fan)
download_mimii(['fan'])

# Для всех четырёх типов:
# download_mimii(['fan', 'pump', 'slider', 'valve'])

## 2. Исследование данных

In [2]:
MACHINE = 'fan'
SNR = 6

files = discover_files(MACHINE, SNR)
print('Machine IDs with normal data:', list(files['normal'].keys()))
print('Machine IDs with abnormal data:', list(files['abnormal'].keys()))

for label in ('normal', 'abnormal'):
    total = sum(len(v) for v in files[label].values())
    print(f'{label}: {total} files across {len(files[label])} machine IDs')

FileNotFoundError: Data directory not found: C:\Users\terrdet\Desktop\ript\data\6_dB_fan

In [ ]:
# Визуализация примера спектрограммы
sample_file = list(files['normal'].values())[0][0]
y_sample = load_audio(sample_file)
wins = window_signal(y_sample)
mel = extract_mel_spectrogram(wins[0])
mfcc = extract_mfcc(wins[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(mel, aspect='auto', origin='lower', cmap='magma')
axes[0].set_title('Log-Mel Spectrogram (Normal)')
axes[0].set_xlabel('Time Frame')
axes[0].set_ylabel('Mel Band')

axes[1].imshow(mfcc, aspect='auto', origin='lower', cmap='coolwarm')
axes[1].set_title('MFCC (Normal)')
axes[1].set_xlabel('Time Frame')
axes[1].set_ylabel('Coefficient')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sample_features.png', dpi=150)
plt.show()

print(f'Audio duration: {len(y_sample)/SAMPLE_RATE:.1f}s')
print(f'Windows per file: {len(wins)}')
print(f'Mel shape: {mel.shape}')
print(f'MFCC shape: {mfcc.shape}')

## 3. Подготовка датасетов

- Тренировка: только нормальные звуки (с аугментацией)
- Валидация: нормальные звуки (для порога аномалий)
- Тест: норма + аномалии

In [3]:
train_ds, val_ds, test_ds, test_labels, test_abnormal_ids = build_datasets(
    MACHINE, snr_db=SNR, augment_train=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Проверим размер входных тензоров
sample_batch = next(iter(train_loader))
if isinstance(sample_batch, (list, tuple)):
    print(f'Input shape: {sample_batch[0].shape}')
else:
    print(f'Input shape: {sample_batch.shape}')

FileNotFoundError: Data directory not found: C:\Users\terrdet\Desktop\ript\data\6_dB_fan

## 4. Модель 1 — Свёрточный автоэнкодер (обнаружение аномалий)

Обучаем только на нормальных данных. Аномалии определяются по ошибке реконструкции.

In [ ]:
model = ConvAutoencoder(latent_dim=LATENT_DIM)
save_path = MODEL_DIR / f'autoencoder_{MACHINE}.pt'

model, history = train_autoencoder(
    model, train_loader, val_loader,
    epochs=EPOCHS, lr=LEARNING_RATE, patience=PATIENCE,
    device=device, save_path=save_path,
)
print(f'Model saved to {save_path}')

In [ ]:
# Кривые обучения
plot_training_history(history, save_name=f'training_history_{MACHINE}.png')

In [ ]:
# Визуализация реконструкции
model.eval()
with torch.no_grad():
    sample = next(iter(test_loader))
    x = sample[0][:1].to(device) if isinstance(sample, (list, tuple)) else sample[:1].to(device)
    x_hat, _ = model(x)
    orig = x[0, 0].cpu().numpy()
    recon = x_hat[0, 0].cpu().numpy()

plot_spectrogram_comparison(orig, recon, save_name=f'spectrogram_comparison_{MACHINE}.png')

### 4.1 Порог аномалии и оценка

In [ ]:
threshold = compute_threshold(model, val_loader, device, quantile=ANOMALY_QUANTILE)

print(f'\n=== Anomaly Detection Results ({MACHINE} @ {SNR}dB) ===')
ad_results = evaluate_anomaly_detection(model, test_loader, test_labels, device, threshold)

# Разделяем ошибки по классам для визуализации
normal_mask = test_labels == 0
abnormal_mask = test_labels == 1
normal_errors = ad_results['errors'][normal_mask]
abnormal_errors = ad_results['errors'][abnormal_mask]

plot_reconstruction_errors(normal_errors, abnormal_errors, threshold,
                           save_name=f'recon_errors_{MACHINE}.png')
plot_roc_curve(test_labels, ad_results['errors'],
               save_name=f'roc_{MACHINE}.png')

## 5. Модель 2 — Классификация типа дефекта

Используем латентные признаки автоэнкодера + MLP классификатор.
Классы формируются по machine_id аномальных записей (разные типы дефектов).

In [ ]:
# Извлекаем латентные признаки для всех тестовых данных
test_features, _ = extract_latent_features(model, test_loader, device)
print(f'Latent features shape: {test_features.shape}')

# Формируем метки для классификации дефектов
# Класс 0 = normal, далее каждый machine_id аномальных записей = отдельный класс
unique_abnormal_ids = sorted(set(aid for aid in test_abnormal_ids if aid is not None))
id_to_class = {aid: i + 1 for i, aid in enumerate(unique_abnormal_ids)}
class_names = ['normal'] + [f'anomaly_{aid}' for aid in unique_abnormal_ids]

classification_labels = []
for i, aid in enumerate(test_abnormal_ids):
    if aid is None:
        classification_labels.append(0)  # normal
    else:
        classification_labels.append(id_to_class[aid])
classification_labels = np.array(classification_labels)

print(f'Classes: {class_names}')
print(f'Class distribution: {np.bincount(classification_labels)}')

In [ ]:
# Разбиваем на train/test для классификатора
from sklearn.model_selection import train_test_split

clf_train_idx, clf_test_idx = train_test_split(
    np.arange(len(classification_labels)),
    test_size=0.3, stratify=classification_labels, random_state=RANDOM_SEED,
)

clf_train_feats = test_features[clf_train_idx]
clf_train_labels = classification_labels[clf_train_idx]
clf_test_feats = test_features[clf_test_idx]
clf_test_labels = classification_labels[clf_test_idx]

num_classes = len(class_names)
classifier = DefectClassifier(latent_dim=LATENT_DIM, num_classes=num_classes)

classifier, clf_history = train_classifier(
    classifier, clf_train_feats, clf_train_labels,
    val_features=clf_test_feats, val_labels=clf_test_labels,
    epochs=80, lr=1e-3, batch_size=64, device=device,
    save_path=MODEL_DIR / f'classifier_{MACHINE}.pt',
)

print(f'\n=== Defect Classification Results ({MACHINE}) ===')
clf_results = evaluate_classifier(
    classifier, clf_test_feats, clf_test_labels, device, class_names,
)

plot_confusion_matrix(clf_results['confusion_matrix'], class_names,
                      save_name=f'confusion_matrix_{MACHINE}.png')

## 6. Визуализация латентного пространства (t-SNE)

In [ ]:
label_names = {i: name for i, name in enumerate(class_names)}
plot_tsne(test_features, classification_labels, label_names=label_names,
          save_name=f'tsne_{MACHINE}.png')

## 7. Устойчивость к шуму (SNR: -6, 0, 6 dB)

Модель обучена на SNR=6dB. Оцениваем на всех трёх уровнях.

In [ ]:
snr_results = evaluate_snr_robustness(
    model, MACHINE, SNR_LEVELS, device, threshold, batch_size=BATCH_SIZE,
)

print('\n=== SNR Robustness Summary ===')
for snr, res in sorted(snr_results.items()):
    print(f'  SNR={snr:3d}dB: AUC={res["auc_roc"]:.4f}, pAUC={res["pauc_01"]:.4f}')

if len(snr_results) > 1:
    plot_snr_comparison(snr_results, save_name=f'snr_comparison_{MACHINE}.png')

## 8. Обобщение на другой тип машины

Обучена на fan → тестируем на pump (cross-machine generalization).

In [ ]:
CROSS_MACHINE = 'pump'

try:
    _, _, cross_test_ds, cross_labels, _ = build_datasets(
        CROSS_MACHINE, snr_db=SNR, augment_train=False,
    )
    cross_loader = DataLoader(cross_test_ds, batch_size=BATCH_SIZE, shuffle=False)

    print(f'\n=== Cross-Machine Generalization ({MACHINE} → {CROSS_MACHINE}) ===')
    cross_results = evaluate_anomaly_detection(
        model, cross_loader, cross_labels, device, threshold,
    )
except FileNotFoundError:
    print(f'Data for {CROSS_MACHINE} not found. Download it first:')
    print(f'  download_mimii(["{CROSS_MACHINE}"])')

## 9. Бонус: One-Class SVM на MFCC

In [ ]:
from src.ocsvm import run_ocsvm_baseline

print(f'=== One-Class SVM Baseline ({MACHINE} @ {SNR}dB) ===')
ocsvm_results = run_ocsvm_baseline(MACHINE, snr_db=SNR)

# Сравнение с автоэнкодером
print('\n=== Comparison ===')
print(f'Autoencoder  AUC: {ad_results["auc_roc"]:.4f}  pAUC: {ad_results.get("pauc_01", "N/A")}')
ae_pauc = partial_auc(test_labels, ad_results['errors'], max_fpr=0.1)
print(f'Autoencoder  AUC: {ad_results["auc_roc"]:.4f}  pAUC: {ae_pauc:.4f}')
print(f'OC-SVM       AUC: {ocsvm_results["auc_roc"]:.4f}  pAUC: {ocsvm_results["pauc_01"]:.4f}')

## 10. Итоговая сводка результатов

In [ ]:
print('=' * 60)
print('ИТОГОВЫЕ РЕЗУЛЬТАТЫ')
print('=' * 60)
print(f'\nМашина: {MACHINE}')
print(f'Основной SNR: {SNR} dB')
print(f'Device: {device}')
print(f'\n--- Обнаружение аномалий (Conv Autoencoder) ---')
print(f'AUC-ROC:         {ad_results["auc_roc"]:.4f}')
print(f'pAUC(FPR≤0.1):   {ae_pauc:.4f}')
print(f'Threshold (q={ANOMALY_QUANTILE}): {threshold:.6f}')

print(f'\n--- Классификация дефектов (MLP) ---')
print(f'Macro-F1:  {clf_results["macro_f1"]:.4f}')
print(f'Accuracy:  {clf_results["accuracy"]:.4f}')
print(f'Classes:   {class_names}')

print(f'\n--- Устойчивость к шуму ---')
for snr, res in sorted(snr_results.items()):
    print(f'  SNR={snr:3d}dB: AUC={res["auc_roc"]:.4f}')

print(f'\n--- Бонус: OC-SVM ---')
print(f'AUC-ROC: {ocsvm_results["auc_roc"]:.4f}')

print('\n' + '=' * 60)
print('Все графики сохранены в:', RESULTS_DIR)
print('Модели сохранены в:', MODEL_DIR)